# BRIO with 32 candidates: candidate generation, preprocessing, training with several seeds

Steps:
1. Check the existing 16-candidate files for gibberish candidates.
2. Small test to choose the diversity penalty for 32 candidates.
3. Generate 32 candidates for test, val and train (resumable, saved on Drive).
4. Preprocess into the BRIO dataset format and save it to Drive as a zip.
5. Train with `--max_num 32`, one run per seed with `--seed`.

During generation, an article with more than 1 candidate at ROUGE = 0 (gibberish) is generated again with a lower diversity penalty. Every such article is written to `<split>.out.regen_log.jsonl` next to the candidate file.

In [1]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


### Env

In [1]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

Cloning into 'BRIO'...
remote: Enumerating objects: 431, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 431 (delta 102), reused 93 (delta 66), pack-reused 291 (from 2)
Receiving objects: 100% (431/431), 15.93 MiB | 8.87 MiB/s, done.
Resolving deltas: 100% (256/256), done.
Mounted at /content/drive
/content/BRIO


In [2]:
# Same env setup as BRIO_1_1.ipynb, then the fine-tuned IndoBART and the Liputan6 source/target files.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("done copying indobart finetuned, start copying dataset...")
! cp /content/drive/MyDrive/BRIO/Dataset/liputan6_converted.zip .
! unzip -q liputan6_converted.zip
! rm liputan6_converted.zip
! cp /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source ./liputan6_converted/canonical
! cp /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target ./liputan6_converted/canonical
import os
os.makedirs('./cache', exist_ok=True)
os.makedirs('./result', exist_ok=True)
print("all done")

Retrieving notices: - done

_libgcc_mutex-0.1    | :   0% 0/1 [00:00<?, ?it/s]
blas-1.0             | :   0% 0/1 [00:00<?, ?it/s]

ca-certificates-2021 | :   0% 0/1 [00:00<?, ?it/s]


intel-openmp-2021.3. | :   0% 0/1 [00:00<?, ?it/s]



ld_impl_linux-64-2.3 | :   0% 0/1 [00:00<?, ?it/s]




libstdcxx-ng-9.3.0   | :   0% 0/1 [00:00<?, ?it/s]





libgomp-9.3.0        | :   0% 0/1 [00:00<?, ?it/s]






mkl-2021.3.0         | :   0% 0/1 [00:00<?, ?it/s]







_openmp_mutex-4.5    | :   0% 0/1 [00:00<?, ?it/s]








libgcc-ng-9.3.0      | :   0% 0/1 [00:00<?, ?it/s]









bzip2-1.0.8          | :   0% 0/1 [00:00<?, ?it/s]










cudatoolkit-11.1.1   | :   0% 0/1 [00:00<?, ?it/s]











giflib-5.2.1         | :   0% 0/1 [00:00<?, ?it/s]












gmp-6.2.1            | :   0% 0/1 [00:00<?, ?it/s]













jpeg-9b              | :   0% 0/1 [00:00<?, ?it/s]














lame-3.100           | :   0% 0/1 [00:00<?, ?it/s]















libffi-3.3           | :   0% 0/1 

### Settings

In [3]:
import os

NUM_CANDS = 32            # candidates per article
GEN_BATCH_SIZE = 8        # articles per generation batch; lower it if generation runs out of GPU memory
DIVERSITY_PENALTY = 0.5   # EDIT after the small test in step 2

# Source/target files (local, unzipped above) and the fine-tuned IndoBART that generates the candidates.
DATA_DIR = "./liputan6_converted/canonical"
MODEL_DIR = "./indobart-liputan6-finetuned"

# Drive folder for the new 32-candidate files, and the zip of the preprocessed dataset.
CAND_DIR = "/content/drive/MyDrive/BRIO/liputan6_candidates_32/canonical"
DATASET_ZIP = "/content/drive/MyDrive/BRIO/Dataset/liputan6_32.zip"

# Existing 16-candidate files, only used by the check in step 1.
OLD_CAND_DIR = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical"

# Checkpoint folder per seed: CKPT_BASE + seed.
CKPT_BASE = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_32_seed"

for split in ["train", "val", "test"]:
    for ext in ["source", "target"]:
        p = os.path.join(DATA_DIR, split + "." + ext)
        print("found  " if os.path.exists(p) else "MISSING", p)

found   ./liputan6_converted/canonical/train.source
found   ./liputan6_converted/canonical/train.target
found   ./liputan6_converted/canonical/val.source
found   ./liputan6_converted/canonical/val.target
found   ./liputan6_converted/canonical/test.source
found   ./liputan6_converted/canonical/test.target


### 1. Check the existing 16-candidate files

This shows how many candidates in the files you already trained on have ROUGE = 0, and at which candidate positions.

In [4]:
# Report for each existing 16-candidate file on Drive (skipped if the file is not there).
for split in ["test", "val", "train"]:
    out_file = os.path.join(OLD_CAND_DIR, split + ".out")
    tgt_file = os.path.join(DATA_DIR, split + ".target")
    if os.path.exists(out_file):
        print("=====", split)
        !conda run --no-capture-output -n env python check_candidates.py --out_file "{out_file}" --tgt_file "{tgt_file}" --num_cands 16 --show 1
    else:
        print("not found:", out_file)

===== test
File: /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.out
Articles: 10972, candidates per article: 16
Candidates with ROUGE = 0: 21927 of 175552 (12.49%)
Articles with more than 1 such candidates: 9983 (90.99%)
Distinct candidates per article (average): 15.8 of 16
Articles by number of ROUGE = 0 candidates: [(0, 939), (1, 50), (2, 8186), (3, 1735), (4, 39), (5, 11), (6, 5), (7, 4), (9, 2), (13, 1)]
ROUGE = 0 candidates by position (0 = first beam group): [(0, 22), (1, 20), (2, 18), (3, 23), (4, 19), (5, 23), (6, 31), (7, 23), (8, 15), (9, 20), (10, 15), (11, 19), (12, 17), (13, 1674), (14, 9992), (15, 9996)]

Article 0, ROUGE = 0 at positions [14, 15]
  reference: kapolda riau baru brigjen pol . johny yodjana bertekad memberantas pelaku penyelundupan kayu di riau . ia berjanji akan menindak tegas pelaku tanpa pa
  candidate:  nip nip sing singosokosok tahunan tahunan awal awal krimin krimin pres pres ef ef kwar kwar pasi pasi karakter karakter pundi pundi peri

### 2. Small test: choose the diversity penalty

With 32 beam groups the diversity penalty adds up over more groups than with 16, so the later candidates may turn into gibberish at 0.5. This cell generates 32 candidates for the first 48 validation articles with three penalty values and reports each one.

How to read the report:
- "Candidates with ROUGE = 0" should be close to 0%. If it is high, the penalty is too strong.
- "Distinct candidates per article" should stay close to 32. If it drops a lot, the penalty is too weak and the candidates are duplicates.

Pick the largest penalty with almost no ROUGE = 0 candidates and set `DIVERSITY_PENALTY` in the Settings cell.

In [30]:
# Generate 32 candidates for 48 validation articles with three penalties, without regeneration, then report.
# GEN_BATCH_SIZE = 4
for penalty in [0.5, 0.2, 0.1]:
    out_file = "./penalty_test/p%s/val.out" % penalty
    print("===== diversity penalty", penalty)
    !conda run --no-capture-output -n env python gen_candidate.py --dataset liputan6 --src_dir "{DATA_DIR}/val.source" --tgt_dir "{out_file}" --model_dir "{MODEL_DIR}" --num_cands {NUM_CANDS} --batch_size {GEN_BATCH_SIZE} --diversity_penalty {penalty} --num_samples 48 --no_rouge_check
    !conda run --no-capture-output -n env python check_candidates.py --out_file "{out_file}" --tgt_file "{DATA_DIR}/val.target" --num_cands {NUM_CANDS} --show 1

===== diversity penalty 0.5
Limiting to first 48 source lines (--num_samples).
Generating 16 candidates per article, diversity penalty 0.5, batch size 4.
0
  1.20 s per article, about 0.0 h left for this split, peak GPU memory 1.6 GB
Final sync to Drive complete.
File: ./penalty_test/p0.5/val.out
Articles: 48, candidates per article: 16
Candidates with ROUGE = 0: 0 of 768 (0.00%)
Articles with more than 1 such candidates: 0 (0.00%)
Distinct candidates per article (average): 15.0 of 16
Articles by number of ROUGE = 0 candidates: [(0, 48)]
ROUGE = 0 candidates by position (0 = first beam group): []
===== diversity penalty 0.2
Limiting to first 48 source lines (--num_samples).
Generating 16 candidates per article, diversity penalty 0.2, batch size 4.
0
  1.14 s per article, about 0.0 h left for this split, peak GPU memory 1.5 GB
Final sync to Drive complete.
File: ./penalty_test/p0.2/val.out
Articles: 48, candidates per article: 16
Candidates with ROUGE = 0: 0 of 768 (0.00%)
Articles with

### 3. Generate 32 candidates

The output is written to `CAND_DIR` on Drive. If Colab disconnects, run the same cell again: it continues from the last article saved on Drive. The train split is the long one (193,883 articles).

In [32]:
# Helper: generate the candidates of one split, with the ROUGE = 0 check and regeneration switched on.
def generate_split(split, sync_every):
    !conda run --no-capture-output -n env python gen_candidate.py --dataset liputan6 --src_dir "{DATA_DIR}/{split}.source" --tgt_dir "{CAND_DIR}/{split}.out" --model_dir "{MODEL_DIR}" --num_cands 32 --batch_size {GEN_BATCH_SIZE} --diversity_penalty {DIVERSITY_PENALTY} --sync_every {sync_every}

print("NUM_CANDS =", NUM_CANDS, "| DIVERSITY_PENALTY =", DIVERSITY_PENALTY)

NUM_CANDS = 16 | DIVERSITY_PENALTY = 0.5


In [ ]:
# Test split (10,972 articles).
generate_split("test", 100)

Generating 32 candidates per article, diversity penalty 0.5, batch size 4.
0
  3.41 s per article, about 10.4 h left for this split, peak GPU memory 3.5 GB
  3.05 s per article, about 9.2 h left for this split, peak GPU memory 3.5 GB
100
  3.10 s per article, about 9.3 h left for this split, peak GPU memory 3.5 GB
200
  3.18 s per article, about 9.4 h left for this split, peak GPU memory 3.5 GB
300
  3.19 s per article, about 9.4 h left for this split, peak GPU memory 3.5 GB
  synced to Drive at batch 100
400
  3.19 s per article, about 9.3 h left for this split, peak GPU memory 3.5 GB
500
  3.26 s per article, about 9.4 h left for this split, peak GPU memory 3.5 GB
600
  3.27 s per article, about 9.3 h left for this split, peak GPU memory 3.5 GB
700
  3.25 s per article, about 9.2 h left for this split, peak GPU memory 3.5 GB
  synced to Drive at batch 200
800
  3.27 s per article, about 9.1 h left for this split, peak GPU memory 3.5 GB
900
  3.34 s per article, about 9.2 h left for t

In [ ]:
# Validation split (10,972 articles).
generate_split("val", 100)

In [ ]:
# Train split (193,883 articles). The file is copied to Drive every 500 batches because it becomes large.
generate_split("train", 500)

In [ ]:
# Report for the new 32-candidate files: ROUGE = 0 candidates per position and distinct candidates.
for split in ["test", "val", "train"]:
    print("=====", split)
    !conda run --no-capture-output -n env python check_candidates.py --out_file "{CAND_DIR}/{split}.out" --tgt_file "{DATA_DIR}/{split}.target" --num_cands {NUM_CANDS} --show 1

In [ ]:
# Articles that were regenerated, and how many of them are still above the limit after all retries.
import pandas as pd
for split in ["test", "val", "train"]:
    log_file = os.path.join(CAND_DIR, split + ".out.regen_log.jsonl")
    if os.path.exists(log_file):
        log = pd.read_json(log_file, lines=True)
        print(split, "| regenerated:", len(log), "| not fixed:", int((~log["fixed"]).sum()))
    else:
        print(split, "| no article needed regeneration")

### 4. Preprocess into the BRIO format

`preprocess.py` reads `.source`, `.target`, `.out` and their `.tokenized` copies from one folder, scores every candidate with ROUGE and writes one json file per article.

In [ ]:
# Put the candidate files next to the source/target files and make the .tokenized copies
# (no Stanford tokenizer for Indonesian, same as before).
import shutil
for split in ["train", "val", "test"]:
    shutil.copyfile(os.path.join(CAND_DIR, split + ".out"), os.path.join(DATA_DIR, split + ".out"))
    for ext in ["source", "target", "out"]:
        shutil.copyfile(os.path.join(DATA_DIR, split + "." + ext), os.path.join(DATA_DIR, split + "." + ext + ".tokenized"))
print("files ready")

In [ ]:
# Build ./liputan6/canonical/{train,val,test} with 32 candidates per article.
for split in ["test", "val", "train"]:
    !conda run --no-capture-output -n env python preprocess.py --src_dir "{DATA_DIR}" --tgt_dir ./liputan6/canonical --split {split} --cand_num {NUM_CANDS} --dataset liputan6 -l

# test.source / test.target are needed by main.py -e
! cp {DATA_DIR}/test.source ./liputan6/canonical/
! cp {DATA_DIR}/test.target ./liputan6/canonical/
! cp {DATA_DIR}/test.target ./liputan6/

In [ ]:
# Save the preprocessed dataset to Drive as one zip, so later sessions can skip steps 2 to 4.
! cd liputan6 && zip -q -r ../liputan6_32.zip .
! cp liputan6_32.zip "{DATASET_ZIP}"
print("saved to", DATASET_ZIP)

### 5. Training with 32 candidates, one run per seed

In a new session, run the env cell, the Settings cell and the restore cell below, then the training cell of the seed you want.

`--seed` and `--max_num` now override `config.py`. Each seed writes to its own checkpoint folder on Drive.

In [20]:
# New session only: restore the preprocessed 32-candidate dataset from Drive into ./liputan6
import zipfile
if not os.path.exists("./liputan6/canonical/train"):
    ! cp "{DATASET_ZIP}" .
    with zipfile.ZipFile("./liputan6_32.zip", "r") as zip_ref:
        zip_ref.extractall("./liputan6")
print("train files:", len(os.listdir("./liputan6/canonical/train")))

train files: 0


In [ ]:
# Seed 1 (the default seed in config.py)
SEED = 970903
NUM_CANDS = 16
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

In [ ]:
# Seed 2
SEED = 70905
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

In [ ]:
# Seed 3
SEED = 81215
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

### Notes

- **Out of GPU memory during training:** 32 candidates use about twice the memory of 16. Add `--batch_size 1 --accumulate_step 16` to the training command. This keeps the same number of articles per optimizer step as the default (batch size 2, accumulate 8).
- **Evaluation with `main.py -e`:** add `--max_num 32` so the reranking evaluation uses all 32 candidates.
- **ROUGE, BERTScore and the Gemini judge:** add the new `model_generation.bin` paths to `BRIO_CKPTS` in `BRIO_bertscore.ipynb` / `BRIO_llm_judge.ipynb`.